# ⚙️ 02 — Feature Engineering

> **Credit Risk ML** · Notebook 2 de 5

**Objetivo:** transformar dados brutos em representações que maximizem a capacidade preditiva dos modelos.

| Etapa | Descrição |
|-------|-----------|
| 2.1 | Limpeza e imputação |
| 2.2 | Criação de features derivadas |
| 2.3 | Validação das novas features |
| 2.4 | Split treino/teste (sem leakage) |
| 2.5 | Normalização |
| 2.6 | Resumo do pipeline de transformação |

---

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor':'#0F1117','axes.facecolor':'#161B22','axes.edgecolor':'#30363D',
    'axes.labelcolor':'#E6EDF3','xtick.color':'#8B949E','ytick.color':'#8B949E',
    'text.color':'#E6EDF3','grid.color':'#21262D','grid.linewidth':0.8,
    'figure.dpi':120,'font.family':'monospace',
})
C = {'good':'#3FB950','bad':'#F78166','neutral':'#58A6FF',
     'accent':'#D2A8FF','warn':'#E3B341','muted':'#8B949E'}

SEED = 42; TEST_SIZE = 0.20
print('✅  Imports OK')

In [ ]:
# Carrega dados (mesma lógica do notebook 01)
DATA_PATH = Path('../data/synthetic/credit_data.csv')
if DATA_PATH.exists():
    df_raw = pd.read_csv(DATA_PATH)
else:
    rng = np.random.default_rng(SEED)
    n   = 50_000
    income      = rng.lognormal(8.5, 0.6, n).round(2)
    loan_amount = (income * rng.uniform(0.5,5.0,n)).round(2)
    loan_tenure = rng.choice([12,24,36,48,60,72],n)
    credit_score= np.clip(rng.normal(650,100,n),300,850).round().astype(int)
    emp_years   = np.clip(rng.exponential(5,n),0,40).round(1)
    dti         = np.clip((loan_amount/loan_tenure)/(income/12),0.01,2.0).round(4)
    log_odds    = -2.5+(-0.006)*credit_score+1.5*dti+0.12*rng.integers(0,10,n)\
                  -0.04*emp_years-0.003*(income/1000)+0.3*(loan_amount/loan_amount.mean())
    default     = rng.binomial(1,1/(1+np.exp(-log_odds))).astype(int)
    df_raw = pd.DataFrame({
        'age':rng.integers(21,70,n),'income':income,'loan_amount':loan_amount,
        'loan_tenure':loan_tenure,'credit_score':credit_score,
        'num_open_accounts':rng.integers(1,15,n),'num_credit_inq':rng.integers(0,10,n),
        'debt_to_income':dti,'employment_years':emp_years,
        'has_mortgage':rng.integers(0,2,n),'has_dependents':rng.integers(0,2,n),
        'default':default,
    })
    for col in ['income','credit_score','employment_years']:
        df_raw.loc[rng.random(n)<0.03, col] = np.nan

print(f'✅  Raw shape: {df_raw.shape} | nulos: {df_raw.isnull().sum().sum()}')

---
### 2.1 Limpeza e Imputação

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Limpeza em 4 passos:
      1. Remove duplicatas
      2. Remove linhas sem target
      3. Imputa nulos pela mediana   ← calculada no próprio conjunto
      4. Clipa outliers (1.5 × IQR)

    NOTA: Em produção, a mediana deve ser calculada APENAS no treino
    e aplicada ao teste via sklearn Pipeline ou transformer customizado.
    """
    df = df.copy()
    n0 = len(df)

    df.drop_duplicates(inplace=True)
    df.dropna(subset=['default'], inplace=True)
    df['default'] = df['default'].astype(int)

    num_cols = df.select_dtypes(include=[np.number]).columns.drop('default')
    medians  = df[num_cols].median()          # ← calculado no conjunto atual
    df[num_cols] = df[num_cols].fillna(medians)

    for col in num_cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr    = q3 - q1
        df[col] = df[col].clip(q1 - 1.5*iqr, q3 + 1.5*iqr)

    print(f'  Removidas: {n0 - len(df)} linhas | Nulos restantes: {df.isnull().sum().sum()}')
    return df

df_clean = clean_data(df_raw)
print(f'✅  Clean shape: {df_clean.shape}')

---
### 2.2 Criação de Features Derivadas

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cria 6 variáveis derivadas baseadas em conhecimento de domínio de crédito.

    Cada feature tem justificativa de negócio clara — não são transformações
    aleatórias, mas variáveis que analistas de crédito realmente usam.
    """
    df = df.copy()

    # ── 1. Razão empréstimo / renda ──────────────────────────────────────────
    # Quanto maior, maior o endividamento relativo
    df['loan_to_income'] = (
        df['loan_amount'] / df['income'].replace(0, np.nan)
    ).fillna(0).clip(0, 20).round(4)

    # ── 2. Comprometimento mensal da renda ───────────────────────────────────
    # Parcela mensal estimada ÷ renda mensal
    # Regra do setor: > 30% é sinal de alerta; > 50% é alto risco
    monthly_pay = df['loan_amount'] / df['loan_tenure'].replace(0, np.nan)
    monthly_inc = df['income'] / 12
    df['monthly_pay_ratio'] = (
        monthly_pay / monthly_inc.replace(0, np.nan)
    ).fillna(0).clip(0, 5).round(4)

    # ── 3. Score de risco composto ───────────────────────────────────────────
    # Combina os 3 indicadores mais correlacionados com default
    # Resultado 0 (baixo risco) → 1 (alto risco)
    credit_norm = 1 - (df['credit_score'] - 300) / 550
    dti_norm    = df['debt_to_income'].clip(0, 2) / 2
    inq_norm    = df['num_credit_inq'] / 10
    df['risk_score'] = (0.5*credit_norm + 0.3*dti_norm + 0.2*inq_norm).round(4)

    # ── 4. Índice de estabilidade financeira ─────────────────────────────────
    # Emprego longo + renda alta = maior estabilidade
    # log1p evita que diferenças enormes de escala dominem
    df['stability_index'] = (
        np.log1p(df['employment_years']) * np.log1p(df['income'] / 1_000)
    ).round(4)

    # ── 5. Flag de alto risco ────────────────────────────────────────────────
    # Score baixo E dívida alta = combinação crítica
    df['high_risk_flag'] = (
        (df['credit_score'] < 580) & (df['debt_to_income'] > 0.5)
    ).astype(int)

    # ── 6. Consultas por conta aberta ────────────────────────────────────────
    # Muitas consultas para poucas contas = pressão financeira
    df['inq_per_account'] = (
        df['num_credit_inq'] / df['num_open_accounts'].replace(0, 1)
    ).round(4)

    return df

df_fe = build_features(df_clean)

new_cols = ['loan_to_income','monthly_pay_ratio','risk_score',
            'stability_index','high_risk_flag','inq_per_account']
print('✅  Features criadas:')
for col in new_cols:
    print(f'   {col:<22} min={df_fe[col].min():.3f}  max={df_fe[col].max():.3f}  '
          f'corr_default={df_fe[col].corr(df_fe["default"]):.3f}')

---
### 2.3 Validação das Novas Features

In [ ]:
# Distribuição das novas features por classe de default
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.patch.set_facecolor('#0F1117')
axes = axes.flatten()

for i, col in enumerate(new_cols):
    ax = axes[i]
    if df_fe[col].nunique() == 2:  # binária
        rates = df_fe.groupby(col)['default'].mean()
        ax.bar(rates.index.astype(str), rates.values,
               color=[C['good'],C['bad']], edgecolor='#0F1117', width=0.4)
        ax.set_ylabel('Taxa de Default')
    else:
        for val, color, label in [(0,C['good'],'Bom'),(1,C['bad'],'Default')]:
            ax.hist(df_fe[df_fe['default']==val][col],
                    bins=40, alpha=0.6, color=color, label=label, density=True)
        ax.legend(fontsize=8, framealpha=0.2)
    corr_val = df_fe[col].corr(df_fe['default'])
    ax.set_title(f'{col}\n(corr={corr_val:.3f})', fontsize=10, color='#E6EDF3')
    ax.grid(alpha=0.2)

plt.suptitle('Novas Features × Default',
             fontsize=14, color=C['accent'], fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Correlação das features novas vs originais com o target
all_feats  = [c for c in df_fe.columns if c != 'default']
corr_target = df_fe[all_feats].corrwith(df_fe['default']).abs().sort_values(ascending=True)

colors_bar = [C['accent'] if c in new_cols else C['neutral'] for c in corr_target.index]

fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('#0F1117')
ax.barh(corr_target.index, corr_target.values, color=colors_bar, edgecolor='#0F1117', alpha=0.9)
ax.set_title('Correlação com Target\n(roxo = features novas)',
             fontsize=13, fontweight='bold', color='#E6EDF3')
ax.set_xlabel('|Pearson r|', fontsize=11)
ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=C['accent'], label='Feature nova'),
                   Patch(color=C['neutral'], label='Feature original')],
          fontsize=9, framealpha=0.3)
plt.tight_layout(); plt.show()

---
### 2.4 Split Treino / Teste (sem Data Leakage)

In [ ]:
TARGET   = 'default'
features = [c for c in df_fe.columns if c != TARGET]

X = df_fe[features]
y = df_fe[TARGET]

# Estratificação: mantém proporção de default em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

print(f'  Treino : {len(X_train):,}  (default={y_train.mean():.1%})')
print(f'  Teste  : {len(X_test):,}  (default={y_test.mean():.1%})')
assert abs(y_train.mean()-y_test.mean()) < 0.01, 'Estratificação falhou'
print('✅  Estratificação verificada')

---
### 2.5 Normalização

In [ ]:
# ⛔ ERRADO (leakage): scaler.fit(X)  — usa info do teste
# ✅ CORRETO:

scaler       = StandardScaler()
X_train_sc   = pd.DataFrame(scaler.fit_transform(X_train),  # fit + transform
                             columns=features, index=X_train.index)
X_test_sc    = pd.DataFrame(scaler.transform(X_test),       # apenas transform
                             columns=features, index=X_test.index)

# Verificação visual: antes × depois
selected = ['income', 'loan_amount', 'credit_score', 'risk_score']
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#0F1117')

for ax, (data, title) in zip(axes, [
    (X_train[selected], 'Antes da Normalização'),
    (X_train_sc[selected], 'Depois (StandardScaler)'),
]):
    data.plot.box(ax=ax, color={
        'boxes':C['neutral'],'medians':C['warn'],'whiskers':C['muted'],'caps':C['muted']
    })
    ax.set_title(title, fontsize=12, color='#E6EDF3')
    ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f'  Média pós-scale  (treino): {X_train_sc.mean().abs().mean():.6f} (≈ 0)')
print(f'  Std  pós-scale  (treino): {X_train_sc.std().mean():.6f} (≈ 1)')

---
### 2.6 Resumo do Pipeline de Transformação

In [ ]:
print('╔══════════════════════════════════════════════════════════╗')
print('║       PIPELINE DE FEATURE ENGINEERING — RESUMO         ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  Features originais  : 11                               ║')
print(f'║  Features derivadas  : 6                                ║')
print(f'║  Features finais     : {len(features):<3}                              ║')
print('║                                                          ║')
print('║  Transformações aplicadas:                              ║')
print('║  ✅ Remoção de duplicatas e nulos no target             ║')
print('║  ✅ Imputação por mediana (robusta a outliers)          ║')
print('║  ✅ Clipping IQR (sem remover outliers — preserva info) ║')
print('║  ✅ 6 features derivadas de domínio de crédito          ║')
print('║  ✅ StandardScaler fit apenas no treino                 ║')
print('║  ✅ Estratificação no split                             ║')
print('║                                                          ║')
print('║  → Próximo: 03_model_baseline.ipynb                    ║')
print('╚══════════════════════════════════════════════════════════╝')

# Exporta artefatos para os notebooks seguintes
import joblib, os
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler,   '../models/scaler.pkl')
joblib.dump({'X_train_sc': X_train_sc, 'X_test_sc': X_test_sc,
             'y_train': y_train, 'y_test': y_test,
             'features': features}, '../models/splits.pkl')
print('\n✅  Artefatos salvos em models/ para uso nos próximos notebooks')